# Israeli Law LLM - Phase 2: Instruction Tuning

Fine-tune the Phase 1 model on ~7,300 Hebrew legal Q&A pairs to create a legal chatbot.

**Setup:** Runtime > Change runtime type > **A100 GPU**

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
from huggingface_hub import login

# Replace with your HuggingFace token (needs write access)
# Get one at: https://huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN_HERE")

## Load Phase 1 Model

In [ ]:
from unsloth import FastLanguageModel

# Load YOUR Phase 1 model (not the base DictaLM)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mufeedh28/dictalm2-israeli-law-merged",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(f"Model loaded: mufeedh28/dictalm2-israeli-law-merged")
print(f"Parameters: {model.num_parameters():,}")

## Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Load Q&A Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "mufeedh28/israeli-law-pretrain",
    data_files={"train": "instructions.jsonl"},
    token=True,
)
train_dataset = dataset["train"]
print(f"Training on {len(train_dataset):,} Q&A pairs")
print(f"\nSample:")
sample = train_dataset[0]["conversations"]
print(f"Q: {sample[0]['content'][:100]}")
print(f"A: {sample[1]['content'][:100]}")

## Format with Chat Template

In [ ]:
# Set up chat template for Mistral/DictaLM
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}[INST] {{ message['content'] }} [/INST]{% elif message['role'] == 'assistant' %}{{ message['content'] }}{{ eos_token }}{% endif %}{% endfor %}"
    print("Set Mistral chat template")
else:
    print(f"Using existing chat template")

def format_conversation(example):
    messages = example["conversations"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = train_dataset.map(format_conversation)
print(f"\nFormatted sample:")
print(train_dataset[0]["text"][:300])

## Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=2,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
        push_to_hub=True,
        hub_model_id="mufeedh28/dictalm2-israeli-law-instruct",
        hub_strategy="every_save",
        hub_private_repo=True,
    ),
)

print(f"Effective batch size: {4 * 4}")
print(f"Model will auto-push to HuggingFace every 200 steps")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}, {max_memory} GB")
print(f"Starting training...\n")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Total steps: {trainer_stats.global_step}")

## Save & Push Merged Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("dictalm2-israeli-law-instruct-lora")
tokenizer.save_pretrained("dictalm2-israeli-law-instruct-lora")

model.push_to_hub("mufeedh28/dictalm2-israeli-law-instruct", token=True)
tokenizer.push_to_hub("mufeedh28/dictalm2-israeli-law-instruct", token=True)
print("Pushed LoRA adapters to HuggingFace")

In [ ]:
# Save and push merged full model (standalone, for open-source release)
model.save_pretrained_merged(
    "dictalm2-israeli-law-instruct-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Saved merged model locally")

model.push_to_hub_merged(
    "mufeedh28/dictalm2-israeli-law-instruct-merged",
    tokenizer,
    save_method="merged_16bit",
    token=True,
)
print("Pushed merged model to: mufeedh28/dictalm2-israeli-law-instruct-merged")

## Export to GGUF (for Ollama)

In [ ]:
# Save as GGUF for local inference with Ollama
model.save_pretrained_gguf(
    "dictalm2-israeli-law-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved GGUF locally")

model.push_to_hub_gguf(
    "mufeedh28/dictalm2-israeli-law-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
    token=True,
)
print("Pushed GGUF to: mufeedh28/dictalm2-israeli-law-GGUF")
print("\nRun locally with: ollama run hf.co/mufeedh28/dictalm2-israeli-law-GGUF")

## Test Chatbot

In [ ]:
FastLanguageModel.for_inference(model)

questions = [
    "מהן זכויות השוכר לפי חוק השכירות?",
    "מה קורה אם מעסיק לא משלם פיצויי פיטורים?",
    "האם ניתן לערער על החלטת בית משפט השלום?",
]

for q in questions:
    messages = [{"role": "user", "content": q}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    # Extract just the answer part
    if "[/INST]" in text:
        answer = text.split("[/INST]")[-1].strip()
    else:
        answer = text[len(prompt):].strip()
    print(f"A: {answer}")